# Codec restoration benchmark

Set `SOURCE` below and Run All. `scripts/run_xp.py` must have filled
`artifacts/xp/` first (the model renders come from a GPU pod).

Four methods against the clean original of one chunk, at three MP3 bitrates:
the SAME round-trip (S and L), Apollo, and A2SB. `input` is the untouched MP3
— the do-nothing floor every method must beat.

SDR counts any waveform difference as error; SI-SNR is the same but forgives
gain. The SAME decoder re-realises phase, which both metrics punish however it
sounds — so read the table together with the spectrograms and the players.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from grooveback import audio as ga

SOURCE = "aerofunk"  # "aerofunk" | "codec" — set, then Run All

XP = Path("../artifacts/xp")
BITRATES = ("64k", "128k", "192k")
ORDER = ("input", "same-s", "same-l", "apollo", "a2sb")

RESULTS = json.loads((XP / "results.json").read_text())[SOURCE]

In [ ]:
print(f"{SOURCE} — SDR / SI-SNR in dB against the original\n")
print(f"{'':10}" + "".join(f"{bitrate:>16}" for bitrate in BITRATES))
for method in ORDER:
    row = ""
    for bitrate in BITRATES:
        scores = RESULTS[bitrate].get(method)
        row += (f"{scores['sdr_db']:>8.1f} /{scores['si_snr_db']:>6.1f}"
                if scores else f"{'--':>16}")
    print(f"{method:10}" + row)

## Spectrograms

In [ ]:
def spectrogram(ax, wav, title):
    audio, sr = ga.load(wav)
    ax.imshow(ga.spectrogram_db(audio), origin="lower", aspect="auto",
              cmap="magma", vmin=-100, vmax=0,
              extent=[0, audio.shape[1] / sr, 0, sr / 2 / 1000])
    ax.set_title(title, fontsize=9)

for bitrate in BITRATES:
    wavs = [("original", XP / SOURCE / "original.wav")] + [
        (method, XP / SOURCE / bitrate / f"{method}.wav") for method in ORDER]
    wavs = [(title, path) for title, path in wavs if path.exists()]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True,
                             sharex=True, sharey=True)
    for ax, (title, path) in zip(axes.flat, wavs):
        spectrogram(ax, path, title)
    for ax in axes.flat[len(wavs):]:
        ax.axis("off")
    fig.suptitle(f"{SOURCE} @ {bitrate}, dBFS, kHz over seconds")
    plt.show()

## Listen

Level-matched to −14 LUFS with one shared headroom gain per set.

In [ ]:
for bitrate in BITRATES:
    print(f"── {bitrate} " + "─" * 40)
    for name in ("original", *ORDER):
        wav = XP / SOURCE / bitrate / "listen" / f"{name}.wav"
        if wav.exists():
            print(name)
            display(Audio(str(wav)))